# Example 10: Cross validation

Cross validation is a powerful technique to evaluate the performance of a model on unseen data. In this example, we will showcase the different cross validation methods available in PortfolioOptimisers.jl and how to use them to evaluate the performance of our portfolio optimization models.

Cross validation can be used as a standalone method to evaluate the performance of a model, or it can be used in conjunction with other techniques like hyperparameter tuning or model selection. They can also be used in [`NestedClustered`]-(@ref) and [`Stacking`]-(@ref) optimisation estimators to optimise the outer estimator on the out-of-sample performance of the inner estimators.

This example will only focus on showcasing the different cross validation methods, with examples on how to use them and what metrics can be computed. Further analysis like plots or grid searches have not been implemented yet, but are the top priority of future development.

In [1]:
using PortfolioOptimisers, PrettyTables
# Format for pretty tables.
tsfmt = (v, i, j) -> begin
    if j == 1
        return Date(v)
    else
        return v
    end
end;
resfmt = (v, i, j) -> begin
    if j == 1
        return v
    else
        return isa(v, Number) ? "$(round(v*100, digits=3)) %" : v
    end
end;

## 1. Setting up

For this example, we will use 5 years of daily data. This is so that we have enough data to perform cross validation on significant amounts of data for both training and testing.

Cross validation cannot have precomputed values like we have done in previous examples. This is because the training and testing sets are generated on the fly, and the performance metrics are computed based on the results of the optimization on these sets.

In [2]:
using CSV, TimeSeries, DataFrames, Clarabel, Statistics

X = TimeArray(CSV.File(joinpath(@__DIR__, "SP500.csv.gz")); timestamp = :Date)[(end - 252 * 5):end]
pretty_table(X[(end - 5):end]; formatters = [tsfmt])

# Compute the returns
rd = prices_to_returns(X)

slv = [Solver(; name = :clarabel1, solver = Clarabel.Optimizer,
              settings = Dict("verbose" => false),
              check_sol = (; allow_local = true, allow_almost = true)),
       Solver(; name = :clarabel2, solver = Clarabel.Optimizer,
              settings = Dict("verbose" => false, "max_step_fraction" => 0.95),
              check_sol = (; allow_local = true, allow_almost = true)),
       Solver(; name = :clarabel3, solver = Clarabel.Optimizer,
              settings = Dict("verbose" => false, "max_step_fraction" => 0.9),
              check_sol = (; allow_local = true, allow_almost = true)),
       Solver(; name = :clarabel4, solver = Clarabel.Optimizer,
              settings = Dict("verbose" => false, "max_step_fraction" => 0.85),
              check_sol = (; allow_local = true, allow_almost = true)),
       Solver(; name = :clarabel5, solver = Clarabel.Optimizer,
              settings = Dict("verbose" => false, "max_step_fraction" => 0.8),
              check_sol = (; allow_local = true, allow_almost = true)),
       Solver(; name = :clarabel6, solver = Clarabel.Optimizer,
              settings = Dict("verbose" => false, "max_step_fraction" => 0.75),
              check_sol = (; allow_local = true, allow_almost = true)),
       Solver(; name = :clarabel7, solver = Clarabel.Optimizer,
              settings = Dict("verbose" => false, "max_step_fraction" => 0.70),
              check_sol = (; allow_local = true, allow_almost = true))];

┌────────────┬─────────┬─────────┬─────────┬─────────┬─────────┬─────────┬──────
│  timestamp │    AAPL │     AMD │     BAC │     BBY │     CVX │      GE │     ⋯
│       Date │ Float64 │ Float64 │ Float64 │ Float64 │ Float64 │ Float64 │ Flo ⋯
├────────────┼─────────┼─────────┼─────────┼─────────┼─────────┼─────────┼──────
│ 2022-12-20 │ 131.916 │   65.05 │  31.729 │  77.371 │ 169.497 │  62.604 │ 310 ⋯
│ 2022-12-21 │ 135.057 │   67.68 │  32.212 │  78.729 │  171.49 │   64.67 │ 314 ⋯
│ 2022-12-22 │ 131.846 │   63.86 │  31.927 │  78.563 │ 168.918 │  63.727 │ 311 ⋯
│ 2022-12-23 │ 131.477 │   64.52 │  32.005 │  79.432 │  174.14 │  63.742 │ 314 ⋯
│ 2022-12-27 │ 129.652 │   63.27 │  32.065 │   79.93 │ 176.329 │  64.561 │ 314 ⋯
│ 2022-12-28 │ 125.674 │   62.57 │  32.301 │  78.279 │ 173.728 │  63.883 │  31 ⋯
└────────────┴─────────┴─────────┴─────────┴─────────┴─────────┴─────────┴──────
                                                              14 columns omitted


For this tutorial we will use the basic [`MeanRisk`]-(@ref) estimator, but the cross validation works for all optimisation estimators, even when computing pareto fronts.

In [3]:
mr = MeanRisk(; opt = JuMPOptimiser(; slv = slv))

MeanRisk
  opt ┼ JuMPOptimiser
      │       pr ┼ EmpiricalPrior
      │          │        ce ┼ PortfolioOptimisersCovariance
      │          │           │   ce ┼ Covariance
      │          │           │      │    me ┼ SimpleExpectedReturns
      │          │           │      │       │     w ┼ nothing
      │          │           │      │       │   idx ┴ nothing
      │          │           │      │    ce ┼ GeneralCovariance
      │          │           │      │       │    ce ┼ SimpleCovariance: SimpleCovariance(true)
      │          │           │      │       │     w ┼ nothing
      │          │           │      │       │   idx ┴ nothing
      │          │           │      │   alg ┴ Full()
      │          │           │   mp ┼ DenoiseDetoneAlgMatrixProcessing
      │          │           │      │     pdm ┼ Posdef
      │          │           │      │         │      alg ┼ UnionAll: Newton
      │          │           │      │         │   kwargs ┴ @NamedTuple{}: NamedTuple()
      │ 

## 2. Cross validation
### 2.1 KFold

The simplest form of cross validation is KFold. This method splits the data into K folds, and then iteratively trains on K-1 folds and tests on the remaining fold. This process is repeated K times, with each fold being used as the test set once.

The `KFold`` indices can be generated independently of the optimisation. Let's say we want to perform 5-fold cross validation, this works out to be roughly one per year.

In [4]:
kfold = KFold(; n = 5)

KFold
             n ┼ Int64: 5
   purged_size ┼ Int64: 0
  embargo_size ┴ Int64: 0


For demonstration purposes we can generate the splits using the [`split`]-(@ref) method. This is not necessary as the cross validation will generate them internally.

In [5]:
kfold_res = split(kfold, rd)
display(kfold_res.train_idx)
display(kfold_res.test_idx)

5-element Vector{Vector{Int64}}:
 [253, 254, 255, 256, 257, 258, 259, 260, 261, 262  …  1251, 1252, 1253, 1254, 1255, 1256, 1257, 1258, 1259, 1260]
 [1, 2, 3, 4, 5, 6, 7, 8, 9, 10  …  1251, 1252, 1253, 1254, 1255, 1256, 1257, 1258, 1259, 1260]
 [1, 2, 3, 4, 5, 6, 7, 8, 9, 10  …  1251, 1252, 1253, 1254, 1255, 1256, 1257, 1258, 1259, 1260]
 [1, 2, 3, 4, 5, 6, 7, 8, 9, 10  …  1251, 1252, 1253, 1254, 1255, 1256, 1257, 1258, 1259, 1260]
 [1, 2, 3, 4, 5, 6, 7, 8, 9, 10  …  999, 1000, 1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008]

5-element Vector{UnitRange{Int64}}:
 1:252
 253:504
 505:756
 757:1008
 1009:1260

Let's perform the cross validation.

In [6]:
kfold_pred = cross_val_predict(mr, rd, kfold)

MultiPeriodPredictionResult
  pred ┼ PredictionResult[PredictionResult
       │   res ┼ MeanRiskResult
       │       │        oe ┼ DataType: DataType
       │       │        pa ┼ ProcessedJuMPOptimiserAttributes
       │       │           │       pr ┼ LowOrderPrior
       │       │           │          │         X ┼ 1008×20 SubArray{Float64, 2, Matrix{Float64}, Tuple{Vector{Int64}, Base.Slice{Base.OneTo{Int64}}}, false}
       │       │           │          │        mu ┼ 20-element Vector{Float64}
       │       │           │          │     sigma ┼ 20×20 Matrix{Float64}
       │       │           │          │      chol ┼ nothing
       │       │           │          │         w ┼ nothing
       │       │           │          │       ens ┼ nothing
       │       │           │          │       kld ┼ nothing
       │       │           │          │        ow ┼ nothing
       │       │           │          │        rr ┼ nothing
       │       │           │          │      f_mu ┼ nothing
  

The result is a [`MultiPeriodPredictionResult`]-(@ref) object, which is a wrapper for a vector of [`PredictionResult`]-(@ref) objects, one for each fold. Each [`PredictionResult`]-(@ref) contains the optimisation result based on the training set, and a [`PredictionReturnsResult`]-(@ref) containing the predicted returns result of the optimised portfolio evaluated on its corresponding test set.

We can individually access the result of each fold by indexing into the `pred` field of the [`MultiPeriodPredictionResult`]-(@ref) object, but we can also directly access via the accessing the `mrd` and `mres` properties, which stand for multi-rd and multi-res. `mrd` concatenates the predicted returns into a single [`PredictionReturnsResult`]-(@ref). Since the embargo and purged sizes are zero, the timestamps of the predicted returns should be the same as the timestamps of the original returns result.

In [7]:
println("isequal(kfold_pred.mrd.ts, rd.ts) = $(isequal(kfold_pred.mrd.ts, rd.ts))")

isequal(kfold_pred.mrd.ts, rd.ts) = true


We can also compute performance metrics (risk measures) on the predicted returns. However, we can only use risk measures that use the returns series as an input. This means [`StandardDeviation`]-(@ref), [`NegativeSkewness`]-(@ref), [`TurnoverRiskMeasure`]-(@ref), [`TrackingRiskMeasure`]-(@ref) with `WeightsTracking`, [`Variance`]-(@ref), [`UncertaintySetVariance`]-(@ref), [`EqualRiskMeasure`]-(@ref), [`ExpectedReturn`]-(@ref) and [`ExpectedReturnRiskRatio`]-(@ref), as well as any risk measure that uses any of these cannot be used. But there are ways around this, for example:

- For the variance and standard deviation, we can use [`LowOrderMoment`]-(@ref) with the appropriate algorithms.
- For [`NegativeSkewness`]-(@ref) we can use [`HighOrderMoment`]-(@ref), or [`Skewness`]-(@ref).
- For [`ExpectedReturn`]-(@ref) and [`ExpectedReturnRiskRatio`]-(@ref) we can use [`MeanReturn`]-(@ref) and [`MeanReturnRiskRatio`]-(@ref) respectively.

Here we will compute the variance.

In [8]:
println("KFold(5) prediction variance = $(expected_risk(LowOrderMoment(; alg = SecondMoment()), kfold_pred))")

KFold(5) prediction variance = 0.00013266913989921423


### 2.2 Combinatorial

The `CombinatorialCrossValidation` method generates all possible combinations of the data into training and testing sets. This method is computationally expensive, but provides a more comprehensive evaluation of the model's performance on unseen data.

There is also a way to compute the optimal number of folds and training folds given a user-defined desired training and test set lengths, as well as the relative weight between the training size and number of test paths.

In [9]:
T = size(rd.X, 1)
target_train_size = 200
target_test_size = 70
n_folds, n_test_folds = optimal_number_folds(T, target_train_size, target_test_size)
cfold = CombinatorialCrossValidation(; n_folds = n_folds, n_test_folds = n_test_folds)

CombinatorialCrossValidation
       n_folds ┼ Int64: 13
  n_test_folds ┼ Int64: 11
   purged_size ┼ Int64: 0
  embargo_size ┴ Int64: 0


Let's see the indices this produces.

In [10]:
cfold_res = split(cfold, rd)

CombinatorialCrossValidationResult
  train_idx ┼ 78-element Vector{Vector{Int64}}
   test_idx ┼ 78-element Vector{Vector{Vector{Int64}}}
   path_ids ┴ 11×78 Matrix{Int64}


Here we have 78 splits, each testing path split into 11 folds. This means we have 78 * 11 = 858 total folds, which is a significant increase from the 5 folds we had in KFold. This is the trade-off for having a more comprehensive evaluation of the model's performance on unseen data.

But it also means we need a way to find a good representative of the predictions in order to evaluate the out of sample performance. First let's perform the cross validation.

There is some nuance with this approach in that the splits do not represent the same number of paths, in fact there are only 66 unique paths, which can be seen from `cfold_res.path_ids`.

In [11]:
cfold_res.path_ids

11×78 Matrix{Int64}:
 1  2  3  4  5  6  7  8  9  10  11  12  …  59  60  61  62  63  64  65  66  66
 1  2  3  4  5  6  7  8  9  10  11  12     59  60  61  62  63  64  65  65  66
 1  2  3  4  5  6  7  8  9  10  11  12     59  60  61  62  63  64  64  65  66
 1  2  3  4  5  6  7  8  9  10  11  12     59  60  61  62  63  63  64  65  66
 1  2  3  4  5  6  7  8  9  10  11  12     59  60  61  62  62  63  64  65  66
 1  2  3  4  5  6  7  8  9  10  11  12  …  59  60  61  61  62  63  64  65  66
 1  2  3  4  5  6  7  8  9  10  11  12     59  60  60  61  62  63  64  65  66
 1  2  3  4  5  6  7  8  9  10   7   8     59  59  60  61  62  63  64  65  66
 1  2  3  4  5  6  4  5  6   6   7   8     58  59  60  61  62  63  64  65  66
 1  2  3  2  3  3  4  5  5   6   7   8     58  59  60  61  62  63  64  65  66
 1  1  1  2  2  3  4  4  5   6   7   7  …  58  59  60  61  62  63  64  65  66

We can now perform the cross validation.

In [12]:
cfold_pred = cross_val_predict(mr, rd, cfold)

PopulationPredictionResult
  pred ┴ 66-element Vector{MultiPeriodPredictionResult{Vector{PredictionResult}, PredictionReturnsResult{SubArray{String, 1, Vector{String}, Tuple{Base.Slice{Base.OneTo{Int64}}}, true}, Vector{Float64}, Nothing, Nothing, Vector{Date}, Nothing, Nothing}, Int64}}


We can see that there are indeed 66 predictions. Each is a valid representative of the out-of-sample performance of the model. However, for evaluating the performance, we can use a sample or the median of the predictions. The median is a good representative of the performance, as it is not affected by outliers, and it is a good measure of central tendency. We can do this with custom function, or a functor of a subtype of [`AbstractCrossValidationScorer`]-(@ref). We've implemented a simple one called [`NearestQuantilePrediction`]-(@ref) which takes the prediction with the nearest quantile to the desired quantile of the distribution of predictions, it defaults to the median.

We will use the risk return ratio of the variance as our performance metric. The paths are sorted according to their expected risk, return based risk measures sort them based on descending order, while true risk measures sort them in ascending order.

In [13]:
sharpe_scorer = NearestQuantilePrediction(;
                                          r = MeanReturnRiskRatio(;
                                                                  rk = LowOrderMoment(;
                                                                                      alg = SecondMoment())))

NearestQuantilePrediction
       r ┼ MeanReturnRiskRatio
         │   rt ┼ MeanReturn
         │      │      w ┼ nothing
         │      │   flag ┴ Bool: false
         │   rk ┼ LowOrderMoment
         │      │   settings ┼ RiskMeasureSettings
         │      │            │   scale ┼ Float64: 1.0
         │      │            │      ub ┼ nothing
         │      │            │     rke ┴ Bool: true
         │      │          w ┼ nothing
         │      │         mu ┼ nothing
         │      │        alg ┼ SecondMoment
         │      │            │     ve ┼ SimpleVariance
         │      │            │        │          me ┼ nothing
         │      │            │        │           w ┼ nothing
         │      │            │        │   corrected ┴ Bool: true
         │      │            │   alg1 ┼ Full()
         │      │            │   alg2 ┴ SquaredSOCRiskExpr()
         │   rf ┴ Float64: 0.0
       q ┼ Float64: 0.5
  kwargs ┴ @NamedTuple{}: NamedTuple()


Scorer is a functor which takes a population as an input and outputs a tuple of the single prediction and the index in the population which matches the desired quantile of the distribution of predictions. In this case, we are using the mean return risk ratio with the variance as the risk measure, and we are looking for the prediction with the nearest quantile to 0.5, which is the median.

In [14]:
median_pred_max_sharpe = sharpe_scorer(cfold_pred)

MultiPeriodPredictionResult
  pred ┼ 13-element Vector{PredictionResult}
   mrd ┼ PredictionReturnsResult
       │     nx ┼ 20-element SubArray{String, 1, Vector{String}, Tuple{Base.Slice{Base.OneTo{Int64}}}, true}
       │      X ┼ 1260-element Vector{Float64}
       │     nf ┼ nothing
       │      F ┼ nothing
       │     ts ┼ 1260-element Vector{Date}
       │     iv ┼ nothing
       │   ivpa ┴ nothing
    id ┴ Int64: 18


The prediction `id` corresponds to the index/path id of the prediction in the population.

In [15]:
median_pred_max_sharpe === cfold_pred.pred[median_pred_max_sharpe.id]

true

Similarty to the KFold, the timestamps of the predicted returns should be the same as the timestamps of the original returns result, since the embargo and purged sizes are zero.

In [16]:
isequal(median_pred_max_sharpe.mrd.ts, rd.ts)

true

We can further verify this by computing the risk return ratio of the variance for all predictions and seeing that the prediction with a risk value closest to the median is indeed the same as the one we found with the scorer. Note that the scorer also filters out predictions whose optimisations failed, so in order to be truly rigorous we'd need to skip NaN values in the array of risks, while keeping the indices aligned, but for demonstration purposes this is sufficient.

In [17]:
sharpe_ratios = expected_risk(MeanReturnRiskRatio(;
                                                  rk = LowOrderMoment(;
                                                                      alg = SecondMoment())),
                              cfold_pred)
argmin(abs.(sharpe_ratios .- median(sharpe_ratios))) == median_pred_max_sharpe.id

true

We can choose any compatible risk measure as outlined above, for demonstration purposes we will now rank them based on the variance.

In [18]:
variance_scorer = NearestQuantilePrediction(; r = LowOrderMoment(; alg = SecondMoment()))
median_pred_min_variance = variance_scorer(cfold_pred)

MultiPeriodPredictionResult
  pred ┼ 13-element Vector{PredictionResult}
   mrd ┼ PredictionReturnsResult
       │     nx ┼ 20-element SubArray{String, 1, Vector{String}, Tuple{Base.Slice{Base.OneTo{Int64}}}, true}
       │      X ┼ 1260-element Vector{Float64}
       │     nf ┼ nothing
       │      F ┼ nothing
       │     ts ┼ 1260-element Vector{Date}
       │     iv ┼ nothing
       │   ivpa ┴ nothing
    id ┴ Int64: 18


Again the id matches the prediction with the nearest quantile to the median of the distribution of predictions.

In [19]:
median_pred_min_variance === cfold_pred.pred[median_pred_min_variance.id]

true

As always, the timestamps match.

In [20]:
isequal(median_pred_min_variance.mrd.ts, rd.ts)

true

### 2.3 WalkForward

We offer two different walkforward estimators, `IndexWalkForward` and `DateWalkForward`. The former splits the data based on the number of observations, while the latter splits the data based on the timestamps, and can be used with Julia's `Dates` module to adjust periods to specific times.

The walkforward method is a more realistic evaluation of the model's performance on unseen data, as it mimics the way the model would be used in practice. It can also dynamically use the previous optimisation weights in constraints and risk measures if so desired.

#### 2.3.1 IndexWalkForward

The simpler estimator is `IndexWalkForward` so we will start with this one.

---

*This notebook was generated using [Literate.jl](https://github.com/fredrikekre/Literate.jl).*